# Hakhel and the little ones: a reception history

This notebook traces how Jewish interpreters received the inclusion of `הטף` (“the little ones”) in Deuteronomy 31:12. Of particular interest is the rabbinic question and answer, `טף למה באין? ... ליתן שכר למביאיהן`: why do the little ones come, and why is merit assigned to those who bring them?

The source data and the interpretive layers remain separate. Sefaria supplies search hits, links, and text. `reception_evidence.csv` is a manually curated chronological reading table. The final discovery tags are a researcher-supplied keyword index for navigating the retrieved texts; Sefaria does not supply them.

Three actual source runs are consolidated: the initial exploratory MCP run, the broader live retrieval of 19 June 2026, and a targeted public MCP retrieval of `Maor VaShemesh, Vayeilech 5` on 6 August 2026. Their configurations and raw responses remain distinct; consolidation does not rewrite them as though they were one execution.


## Table of content (ToC)<a class="anchor" id="TOC"></a>
* <a href="#bullet1">1 - Method and limits</a>
* <a href="#bullet2">2 - MCP helpers</a>
* <a href="#bullet3">3 - Discover the reception corpus</a>
* <a href="#bullet4">4 - Follow direct links from the anchor verse</a>
* <a href="#bullet5">5 - Retrieve full texts for close reading</a>
* <a href="#bullet6">6 - Curated chronological map</a>
* <a href="#bullet7">7 - Reception-history synthesis</a>
    * <a href="#bullet7x1">7.1 - The rabbinic problem is not whether children are present</a>
    * <a href="#bullet7x2">7.2 - The saying becomes portable</a>
    * <a href="#bullet7x3">7.3 - Medieval interpreters reopen the scriptural rationale</a>
    * <a href="#bullet7x4">7.4 - Later interpreters differentiate kinds of children</a>
    * <a href="#bullet7x5">7.5 - Synagogue reception produces extension and resistance</a>
    * <a href="#bullet7x6">7.6 - Hasidic readings restore efficacy to the child</a>
    * <a href="#bullet7x7">7.7 - Modern readings foreground intergenerational covenant</a>
* <a href="#bullet8">8 - Main conclusion</a>
* <a href="#bullet9">9 - Export reproducible results</a>
    * <a href="#bullet9x1">9.1 - Consolidate the recorded Sefaria runs</a>
* <a href="#bullet10">10 - Required libraries</a>
* <a href="#bullet11">11 - Notebook details</a>


# 1 - Method and limits<a class="anchor" id="bullet1"></a>
##### [Back to ToC](#TOC)

For this purpose we use three Sefaria MCP operations:

- keyword search to discover reception witnesses;
- link retrieval from Deuteronomy 31:12 to identify direct commentary;
- full-text retrieval for close reading.

By default this notebook rebuilds its tables from the stored, dated source runs. Set `RUN_LIVE_RETRIEVAL = True` only when a new live Sefaria run is intended. Search indexes and available versions can change, so a new run should receive a new date rather than silently replacing the provenance of an older one.

The resulting chronology is a reception map, not a claim that Sefaria contains every relevant source. Search results should be checked against critical editions when textual variants or dating matter.


In [1]:
# Dependencies are listed in reception_history/requirements.txt.
# Uncomment the next line only when preparing a new environment.
# %pip install -q -r "requirements.txt"


In [2]:
# Method, paths, and recorded-run mode
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import pandas as pd
from tqdm.auto import tqdm
from mcp import ClientSession
from mcp.client.sse import sse_client

MCP_SERVER_URL = "https://mcp.sefaria.org/sse"
RUN_LIVE_RETRIEVAL = False

HERE = Path.cwd()
if not (HERE / "results").exists() and (HERE / "reception_history").exists():
    HERE = HERE / "reception_history"
RESULTS = HERE / "results"
RESULTS.mkdir(exist_ok=True)

strategy = json.loads((RESULTS / "search_queries.json").read_text(encoding="utf-8"))
SEARCH_QUERIES = strategy["queries"]
SEED_REFS = strategy["seed_references"]

print("Mode:", "new live retrieval" if RUN_LIVE_RETRIEVAL else "rebuild from stored source runs")
print("Recorded live-search queries:", len(SEARCH_QUERIES))


Mode: rebuild from stored source runs
Recorded live-search queries: 8


# 2 - Handle MCP responses<a class="anchor" id="bullet2"></a>
##### [Back to ToC](#TOC)

In [3]:
# Handle MCP responses
def decode_result(result: Any) -> Any:
    content = getattr(result, "content", result)
    if isinstance(content, list) and content:
        content = getattr(content[0], "text", content[0])
    if isinstance(content, str):
        try:
            payload = json.loads(content)
        except json.JSONDecodeError:
            return content
        if isinstance(payload, dict) and "result" in payload:
            inner = payload["result"]
            if isinstance(inner, str):
                try:
                    return json.loads(inner)
                except json.JSONDecodeError:
                    return inner
            return inner
        return payload
    return content


async def call_tool(name: str, arguments: dict[str, Any]) -> Any:
    async with sse_client(MCP_SERVER_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return decode_result(await session.call_tool(name, arguments))


async def list_tools() -> list[dict[str, Any]]:
    async with sse_client(MCP_SERVER_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            response = await session.list_tools()
            return [tool.model_dump() for tool in response.tools]

# 3 - Discover the reception corpus<a class="anchor" id="bullet3"></a>
##### [Back to ToC](#TOC)

In [4]:
# Discover or reload the reception corpus
def result_rows(payload: Any, query: str) -> list[dict[str, Any]]:
    """Normalize result objects while retaining their search query."""
    if isinstance(payload, dict):
        payload = payload.get("results", payload.get("hits", []))
    rows = []
    for item in payload or []:
        row = dict(item)
        row["search_query"] = query
        rows.append(row)
    return rows


if RUN_LIVE_RETRIEVAL:
    raw_searches = []
    rows = []
    for query in tqdm(SEARCH_QUERIES, desc="Sefaria searches"):
        payload = await call_tool("text_search", {"query": query, "size": 50})
        raw_searches.append({"query": query, "payload": payload})
        rows.extend(result_rows(payload, query))

    search_df = pd.DataFrame(rows)
    if not search_df.empty and "ref" in search_df:
        search_df = search_df.drop_duplicates(subset=["ref"]).reset_index(drop=True)
else:
    raw_searches = [
        json.loads(line)
        for line in (RESULTS / "raw_search_payloads.jsonl").read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    assert [item["query"] for item in raw_searches] == SEARCH_QUERIES, (
        "search_queries.json no longer describes the stored live run"
    )
    search_df = pd.read_csv(RESULTS / "live_search_results.csv")

search_df.head(20)


,ref,categories,text_snippet,search_query
0,Deuteronomy 31:12,"['Tanakh', 'Torah']",<b>הַקְהֵל</b> <b>אֶת</b>־<b>הָעָם</b> <b>הָאֲ...,הקהל את העם האנשים והנשים והטף
1,Chagigah 3a:17,"['Talmud', 'Bavli', 'Seder Moed']",<b>הקהל</b> <b>את</b> <b>העם</b> <b>האנשים</b>...,הקהל את העם האנשים והנשים והטף
2,"Mishneh Torah, Festival Offering 3:1","['Halakhah', 'Mishneh Torah', 'Sefer Korbanot']",מִצְוַת עֲשֵׂה לְהַקְהִיל כָּל יִשְׂרָאֵל <b>א...,הקהל את העם האנשים והנשים והטף
3,Chagigah 3a:2,"['Talmud', 'Bavli', 'Seder Moed']",לענין ראיה גמר ראיה ראיה <b>מהקהל</b> דכתיב <b...,הקהל את העם האנשים והנשים והטף
4,Sefer HaChinukh 612:1,"['Halakhah', 'Sifrei Mitzvot']",לְהַקְהִיל כָּל יִשְׂרָאֵל בְּחַג הַסֻּכּוֹת –...,הקהל את העם האנשים והנשים והטף
5,Jerusalem Talmud Sotah 3:4:6,"['Talmud', 'Yerushalmi', 'Seder Nashim']",וְהִקְבִילוּ <b>אֶת</b> רִבִּי יְהוֹשֻׁעַ בִּב...,הקהל את העם האנשים והנשים והטף
6,Sefer Chasidim 313:1,"['Halakhah', 'Rishonim']",כאלו מלמדה תיפלות זהו עומק תלמוד וטעמי המצות ו...,הקהל את העם האנשים והנשים והטף
7,Bamidbar Rabbah 14:4,"['Midrash', 'Aggadah', 'Midrash Rabbah']",עַל פִּי כֵן אִי אֶפְשָׁר לְבֵית הַמִּדְרָשׁ ב...,הקהל את העם האנשים והנשים והטף
8,Avot DeRabbi Natan 18:2,"['Talmud', 'Bavli', 'Minor Tractates']",אמר להם ובמה היתה ההגדה היום אמרו בפרשת <b>הקה...,הקהל את העם האנשים והנשים והטף
9,"Abudarham, Pesach, Festival Prayers 12","['Halakhah', 'Rishonim']",ועוד טעם אחר לפי שאבותינו לא קבלו <b>את</b> הת...,הקהל את העם האנשים והנשים והטף


# 4 - Follow direct links from the anchor verse<a class="anchor" id="bullet4"></a>
##### [Back to ToC](#TOC)

Direct links reveal a second reception channel that keyword searches can
miss: commentaries that explain the verse without repeating the standard
phrase verbatim.

In [5]:
# Follow or reload direct links from the anchor verse
if RUN_LIVE_RETRIEVAL:
    linked = await call_tool(
        "get_links_between_texts",
        {"reference": "Deuteronomy 31:12", "with_text": "1"},
    )
    links_df = pd.DataFrame(linked or [])
else:
    links_df = pd.read_csv(RESULTS / "anchor_links.csv")

links_df[["ref", "type", "category"]].head(30)


,ref,type,category
0,Ramban on Deuteronomy 31:12:1,commentary,Commentary
1,Rashi on Deuteronomy 31:12:1,commentary,Commentary
2,Rashi on Deuteronomy 31:12:2,commentary,Commentary
3,Ibn Ezra on Deuteronomy 31:12:1,commentary,Commentary
4,Ibn Ezra on Deuteronomy 31:12:2,commentary,Commentary
5,Onkelos Deuteronomy 31:12,targum,Targum
6,Targum Jonathan on Deuteronomy 31:12,targum,Targum
7,Kli Yakar on Deuteronomy 31:12:1,commentary,Commentary
8,Kli Yakar on Deuteronomy 31:12:2,commentary,Commentary
9,Kli Yakar on Deuteronomy 31:12:3,commentary,Commentary


# 5 - Retrieve full texts for close reading<a class="anchor" id="bullet5"></a>
##### [Back to ToC](#TOC)

In [6]:
# Retrieve or reload full texts for close reading
def strip_html(value: Any) -> str:
    if isinstance(value, list):
        return " ".join(strip_html(v) for v in value)
    return re.sub(r"<[^>]+>", "", str(value or "")).strip()


def normalize_text(ref: str, payload: Any) -> dict[str, Any]:
    """Keep the first available Hebrew and English versions."""
    versions = payload.get("versions", []) if isinstance(payload, dict) else []
    row = {"ref": payload.get("ref", ref) if isinstance(payload, dict) else ref}
    for version in versions:
        language = version.get("languageFamilyName", "").lower()
        if language == "hebrew" and "hebrew" not in row:
            row["hebrew"] = strip_html(version.get("text"))
        if language == "english" and "english" not in row:
            row["english"] = strip_html(version.get("text"))
    return row


if RUN_LIVE_RETRIEVAL:
    discovered = [] if search_df.empty else search_df["ref"].dropna().tolist()
    linked_refs = [] if links_df.empty else links_df["ref"].dropna().tolist()
    refs = list(dict.fromkeys(SEED_REFS + discovered + linked_refs))

    texts = []
    for ref in tqdm(refs, desc="Full texts"):
        try:
            payload = await call_tool(
                "get_text", {"reference": ref, "version_language": "both"}
            )
            texts.append(normalize_text(ref, payload))
        except Exception as exc:
            texts.append({"ref": ref, "error": repr(exc)})
    texts_df = pd.DataFrame(texts)
else:
    texts_df = pd.read_csv(RESULTS / "full_texts.csv")

texts_df.head(20)


,ref,english,hebrew
0,Deuteronomy 31:10-13,And Moses instructed them as follows: Every se...,וַיְצַ֥ו מֹשֶׁ֖ה אוֹתָ֣ם לֵאמֹ֑ר מִקֵּ֣ץ&thins...
1,Tosefta Sotah 7:6,It so happened with Rabbi Yochanan ben Berokah...,מעשה בר' יוחנן בן ברוקה ור' אלעזר &lt;בן&gt; ח...
2,Jerusalem Talmud Chagigah 1:1:3,"7The same text is in Soṭah3:4 (Notes 107–109, ...",דְּתַנֵּי. מַעֲשֶׂה בְרִבִּי יוֹחָנַן בֶּן בְּ...
3,Chagigah 3a:17,They said to him that Rabbi Elazar ben Azarya ...,״הַקְהֵל אֶת הָעָם הָאֲנָשִׁים וְהַנָּשִׁים וְ...
4,Mishnah Sotah 7:8,How is the portion of the Torah that is read b...,פָּרָשַׁת הַמֶּלֶךְ כֵּיצַד. מוֹצָאֵי יוֹם טוֹ...
5,"Mishneh Torah, Festival Offering 3",It is a positive commandment to gather togethe...,מִצְוַת עֲשֵׂה לְהַקְהִיל כָּל יִשְׂרָאֵל אֲנָ...
6,Rashi on Deuteronomy 31:12,"האנשים [ASSEMBLE THE PEOPLE,] MEN [WOMEN, AND ...",האנשים. לִלְמֹד: והנשים. לִשְׁמֹעַ: והטף. לָמּ...
7,"Abarbanel on Torah, Deuteronomy 31:9:5",NaN,וצוה הקהל את העם האנשים והנשים והטף. והנראה כד...
8,Kli Yakar on Deuteronomy 31:12,"Gather the people, the men, and the women, and...",הַקְהֵל אֶת הָעָם הָאֲנָשִׁים וְהַנָּשִׁים וְה...
9,"Sefat Emet, Deuteronomy, Vayeilech 3:2",The Gemara (Chagigah 3a) explains that childre...,"טף למה באין לתת שכר למביאיהן. וקשה ממ""נ אי אין..."


# 6 - Curated chronological map<a class="anchor" id="bullet6"></a>
##### [Back to ToC](#TOC)

The retrieval is broad. This table is the close-reading layer: representative witnesses are ordered by reception period and assigned a dominant interpretive move. Its `reference` values use canonical references present in the consolidated Sefaria text table. The categories, chronology, and interpretive descriptions are researcher-supplied and remain open to revision.


In [7]:
# Curated chronological map
evidence = pd.read_csv(RESULTS / "reception_evidence.csv")
evidence

,period,date_range,reference,genre,reception_move,role_of_the_little_ones,theme
0,Biblical base,textual starting point,Deuteronomy 31:10-13,Torah,Commands a septennial public Torah reading bef...,They are explicitly present and 31:13 associat...,covenant inclusion
1,Early rabbinic,2nd-3rd c.,Tosefta Sotah 7:6,Tosefta,Frames the inclusion as a deliberate exegetica...,Their attendance produces reward for those who...,reward to bringers
2,Early rabbinic,3rd-6th c.,Jerusalem Talmud Chagigah 1:1:3,Yerushalmi,Transmits the Elazar ben Azariah interpretatio...,Their presence merits the adults who bring them.,reward to bringers
3,Early rabbinic,3rd-6th c.,Jerusalem Talmud Chagigah 1:1:2,Yerushalmi,Uses Hakhel to argue that a minor can be oblig...,The category טף becomes a legal analogy for mi...,minor obligation
4,Early rabbinic,3rd-6th c.,Chagigah 3a:17,Bavli,Canonizes the question and answer and calls th...,They come so reward may be given to those who ...,reward to bringers
5,Early rabbinic,3rd-7th c.,Avot DeRabbi Natan 18:2,Minor tractate,Retells the study-house narrative and preserve...,Their attendance rewards their bringers.,reward to bringers
6,Early rabbinic,late antique/early medieval,Tractate Soferim 18:6,Minor tractate,Extends the Hakhel saying into synagogue custo...,Attendance habituates children and rewards bot...,synagogue formation
7,Targumic,late antique/medieval,Targum Jonathan on Deuteronomy 31:12,Targum,Builds the rabbinic differentiation directly i...,Children share in the benefit or reward of tho...,translation as interpretation
8,Medieval,11th c.,Rashi on Deuteronomy 31:12,Biblical commentary,Makes the Bavli's three-part division the defa...,They are present to bring merit to those who c...,reward to bringers
9,Medieval,12th c.,Ibn Ezra on Deuteronomy 31:12:3,Biblical commentary,Reads hearing as the trigger for questioning a...,Young children can learn through hearing and a...,child learning


In [8]:
# Summarize the curated evidence by period and theme
pd.crosstab(evidence["period"], evidence["theme"])

theme,Sinai reenactment,age differentiation,age threshold,child learning,communal solidarity,counter-reception,covenant inclusion,covenant renewal,formation,inclusive hearing,minor obligation,reward to bringers,spiritual imprint,synagogue formation,translation as interpretation
period,,,,,,,,,,,,,,,
Biblical base,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
Early modern,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0
Early rabbinic,0,0,0,0,0,0,0,0,0,0,1,4,0,1,0
Hasidic,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0
Late medieval,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
Medieval,1,0,0,1,0,0,0,0,1,1,0,1,0,0,0
Modern,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
Modern halakhic,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
Targumic,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


# 7 - Reception-history synthesis<a class="anchor" id="bullet7"></a>
##### [Back to ToC](#TOC)

The links in this section point to Sefaria locations represented in the evidence package. They open the text fragment used for the corresponding observation, rather than a general search page. Where the stored evidence is a passage range, the link may narrow that range to the particular verse under discussion.

## 7.1 - The rabbinic problem is not whether children are present<a class="anchor" id="bullet7x1"></a>
##### [Back to ToC](#TOC)

[Deuteronomy 31:10–13](https://www.sefaria.org/Deuteronomy.31.10-13) already commands their presence. The [early rabbinic
intervention](https://www.sefaria.org/Chagigah.3a.17) asks what *purpose* that presence serves when infants or
very young children cannot perform the stated acts of learning. The
answer relocates efficacy from child to adult: the bringing itself
becomes meritorious.

## 7.2 - The saying becomes portable<a class="anchor" id="bullet7x2"></a>
##### [Back to ToC](#TOC)

The [Bavli's memorable three-part formula](https://www.sefaria.org/Chagigah.3a.17) moves into [Targum Jonathan](https://www.sefaria.org/Targum_Jonathan_on_Deuteronomy.31.12), [Rashi](https://www.sefaria.org/Rashi_on_Deuteronomy.31.12),
the [Mishneh Torah](https://www.sefaria.org/Mishneh_Torah%2C_Festival_Offering.3), the [Sefer HaChinukh](https://www.sefaria.org/Sefer_HaChinukh.612.1), and later commentary. It then supports arguments about
women's Torah hearing, minors' ritual participation, parental duty, and
the social meaning of public worship.

## 7.3 - Medieval interpreters reopen the scriptural rationale<a class="anchor" id="bullet7x3"></a>
##### [Back to ToC](#TOC)

[Ibn Ezra](https://www.sefaria.org/Ibn_Ezra_on_Deuteronomy.31.12.3) and especially [Abarbanel](https://www.sefaria.org/Abarbanel_on_Torah%2C_Deuteronomy.31.9.5) resist reducing children to passive
occasions for adult reward. [Deuteronomy 31:13](https://www.sefaria.org/Deuteronomy.31.13) says that children who do
not yet know will hear and learn. This creates a durable tension between
the rabbinic reward motif and the verse's educational teleology.

## 7.4 - Later interpreters differentiate kinds of children<a class="anchor" id="bullet7x4"></a>
##### [Back to ToC](#TOC)

[Kli Yakar](https://www.sefaria.org/Kli_Yakar_on_Deuteronomy.31.12) and the [Minchat Chinukh](https://www.sefaria.org/Minchat_Chinukh.612.1.1) divide `טף` by developmental stage:
children capable of education can benefit directly, while younger
children raise questions about the adult obligation to bring, the lower
age boundary, and the meaning of reward.

## 7.5 - Synagogue reception produces extension and resistance<a class="anchor" id="bullet7x5"></a>
##### [Back to ToC](#TOC)

[Tractate Soferim](https://www.sefaria.org/Tractate_Soferim.18.6) turns Hakhel into precedent for bringing young girls to
synagogue. The [*Shenei Luchot HaBerit*](https://www.sefaria.org/Shenei_Luchot_HaBerit%2C_Aseret_HaDibrot%2C_Tamid%2C_Ner_Mitzva.130) preserves a counter-reading: if
children disrupt prayer and desecrate the space, the bringer should
expect punishment rather than reward.

## 7.6 - Hasidic readings restore efficacy to the child<a class="anchor" id="bullet7x6"></a>
##### [Back to ToC](#TOC)

[Sefat Emet](https://www.sefaria.org/Sefat_Emet%2C_Deuteronomy%2C_Vayeilech.3.2) argues that children can receive an inward illumination even
without conceptual understanding. [Maor VaShemesh](https://www.sefaria.org/Maor_VaShemesh%2C_Vayeilech.5) places them within the
love and unity of Israel that makes Torah hearing possible.

## 7.7 - Modern readings foreground intergenerational covenant<a class="anchor" id="bullet7x7"></a>
##### [Back to ToC](#TOC)

[A modern reading retained in the evidence package](https://www.sefaria.org/Covenant_and_Conversation%3B_Deuteronomy%3B_Renewal_of_the_Sinai_Covenant%2C_Vayeilech%2C_To_Renew_Our_Days.6) describes Hakhel as national covenant
renewal and inclusive public education. The little ones signify that
Torah belongs to the next generation before they can fully understand
or consent to it.

# 8 - Main conclusion<a class="anchor" id="bullet8"></a>
##### [Back to ToC](#TOC)

The reception history supplies four recurring answers to why the little
ones come:

1. **adult merit** — bringing them is itself rewarded;
2. **formation** — repeated presence habituates future participants;
3. **direct reception** — children hear, learn, or receive a spiritual
   impression according to their capacity;
4. **corporate identity** — their presence makes the assembly visibly
   intergenerational and covenantal.

These answers coexist. The productive tension is between `טף` as
dependents brought by adults and `טף` as genuine addressees of Torah.

# 9 - Export reproducible results<a class="anchor" id="bullet9"></a>
##### [Back to ToC](#TOC)

In [9]:
# Export a new live run, or confirm the stored run inputs
if RUN_LIVE_RETRIEVAL:
    search_df.to_csv(RESULTS / "live_search_results.csv", index=False, encoding="utf-8-sig")
    links_df.to_csv(RESULTS / "anchor_links.csv", index=False, encoding="utf-8-sig")
    texts_df.to_csv(RESULTS / "full_texts.csv", index=False, encoding="utf-8-sig")

    with (RESULTS / "raw_search_payloads.jsonl").open("w", encoding="utf-8") as handle:
        for item in raw_searches:
            handle.write(json.dumps(item, ensure_ascii=False) + "\n")

    tools = await list_tools()
    (RESULTS / "mcp_tool_inventory.json").write_text(
        json.dumps(tools, ensure_ascii=False, indent=2), encoding="utf-8"
    )
else:
    tools = json.loads((RESULTS / "mcp_tool_inventory.json").read_text(encoding="utf-8"))
    assert (RESULTS / "live_search_results.csv").exists()
    assert (RESULTS / "anchor_links.csv").exists()
    assert (RESULTS / "full_texts.csv").exists()
    assert (RESULTS / "raw_search_payloads.jsonl").exists()

print("Stored source run is complete; live files were not replaced." if not RUN_LIVE_RETRIEVAL else "New live run exported.")


Stored source run is complete; live files were not replaced.


## 9.1 - Consolidate the recorded Sefaria runs<a class="anchor" id="bullet9x1"></a>
##### [Back to ToC](#TOC)

The repository contains three separately identified source runs. Their raw responses and dates remain distinct. Consolidation first writes a query-hit history, retaining repeated hits and every query/reference relationship, and only then creates a deduplicated reading table.

The full-text history likewise retains one row per source run. The targeted 6 August supplement supplies the Maor VaShemesh witness which was missing from the earlier broad retrieval. The lightweight discovery tags remain visible keyword rules and must not be confused with the manually curated reception-historical table.


In [10]:
# Consolidate the actual recorded Sefaria source runs
INITIAL_RUN = RESULTS / "source_runs" / "initial_sefaria_mcp"
SUPPLEMENT_RUN = RESULTS / "source_runs" / "targeted_supplement_2026-08-06"


def joined_unique(values: pd.Series) -> str:
    """Join non-empty values without changing first-seen order."""
    seen = []
    for value in values.dropna().astype(str):
        value = value.strip()
        if value and value not in seen:
            seen.append(value)
    return " | ".join(seen)


def first_nonempty(values: pd.Series) -> str:
    """Take the first substantive value from the preferred source order."""
    for value in values.dropna():
        text = str(value).strip()
        if text:
            return text
    return ""


def query_hit_rows(path: Path, source_run: str) -> pd.DataFrame:
    """Normalize every raw hit before any reference-level deduplication."""
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue
        item = json.loads(line)
        for row in result_rows(item.get("payload"), item["query"]):
            rows.append(
                {
                    "source_run": source_run,
                    "ref": row.get("ref", ""),
                    "categories": row.get("categories", ""),
                    "text_snippet": row.get("text_snippet", ""),
                    "search_query": row.get("search_query", ""),
                }
            )
    return pd.DataFrame(rows)


current_search_rows = query_hit_rows(
    RESULTS / "raw_search_payloads.jsonl", "reception_history_live_2026-06-19"
)
initial_search_rows = query_hit_rows(
    INITIAL_RUN / "hakhel_taf_raw_search_payloads.jsonl", "initial_sefaria_mcp"
)
search_source_rows = pd.concat(
    [current_search_rows, initial_search_rows], ignore_index=True, sort=False
)
search_source_rows.to_csv(
    RESULTS / "consolidated_search_source_rows.csv",
    index=False,
    encoding="utf-8-sig",
)

consolidated_search = (
    search_source_rows.groupby("ref", as_index=False, sort=True)
    .agg(
        source_runs=("source_run", joined_unique),
        search_queries=("search_query", joined_unique),
        categories=("categories", first_nonempty),
        text_snippet=("text_snippet", first_nonempty),
    )
)
consolidated_search.to_csv(
    RESULTS / "consolidated_search_references.csv",
    index=False,
    encoding="utf-8-sig",
)

current_text_rows = texts_df.copy()
current_text_rows.insert(0, "source_run", "reception_history_live_2026-06-19")
initial_text_rows = pd.read_csv(INITIAL_RUN / "hakhel_taf_full_texts.csv")
initial_text_rows.insert(0, "source_run", "initial_sefaria_mcp")
supplement_text_rows = pd.read_csv(
    SUPPLEMENT_RUN / "maor_vashemesh_vayeilech_5_full_text.csv"
)
supplement_text_rows.insert(0, "source_run", "targeted_supplement_2026-08-06")

full_text_source_rows = pd.concat(
    [current_text_rows, initial_text_rows, supplement_text_rows],
    ignore_index=True,
    sort=False,
)
full_text_source_rows.to_csv(
    RESULTS / "consolidated_full_text_source_rows.csv",
    index=False,
    encoding="utf-8-sig",
)

value_columns = [
    column
    for column in full_text_source_rows.columns
    if column not in {"ref", "source_run"}
]
consolidated_rows = []
for reference, group in full_text_source_rows.groupby("ref", sort=True):
    row = {"ref": reference, "source_runs": joined_unique(group["source_run"])}
    for column in value_columns:
        row[column] = first_nonempty(group[column])
    consolidated_rows.append(row)

consolidated_texts = pd.DataFrame(consolidated_rows)


def discovery_tags(row: pd.Series) -> str:
    """Apply the earlier notebook's visible keyword navigation rules."""
    haystack = " ".join(
        str(row.get(column, ""))
        for column in ["ref", "hebrew", "english", "raw_text"]
    )
    tags = []
    if any(term in haystack for term in ["Deuteronomy 31", "הקהל את העם", "האנשים והנשים והטף"]):
        tags.append("Torah instruction")
    if any(term in haystack for term in ["טף למה", "ליתן שכר", "bring them", "bringing them"]):
        tags.append("rabbinic reason for children")
    if any(term in haystack for term in ["חגיגה", "Chagigah"]):
        tags.append("Chagigah discussion")
    if any(term in haystack for term in ["קידושין", "Kiddushin", "time-bound"]):
        tags.append("mitzvah obligation discussion")
    return "; ".join(tags) or "related reference"


consolidated_texts["discovery_tags"] = consolidated_texts.apply(
    discovery_tags, axis=1
)
consolidated_texts.to_csv(
    RESULTS / "consolidated_full_texts.csv",
    index=False,
    encoding="utf-8-sig",
)

curated_references = set(evidence["reference"])
stored_text_references = set(consolidated_texts["ref"])
missing_curated_references = sorted(curated_references - stored_text_references)
assert not missing_curated_references, (
    f"Curated references without stored Sefaria text: {missing_curated_references}"
)

audit_path = RESULTS / "audit.json"
audit = json.loads(audit_path.read_text(encoding="utf-8"))
audit.update(
    {
        "curated_evidence_rows": len(evidence),
        "configured_live_search_queries": len(SEARCH_QUERIES),
        "current_raw_query_hits": len(current_search_rows),
        "initial_raw_query_hits": len(initial_search_rows),
        "consolidated_search_source_rows": len(search_source_rows),
        "consolidated_search_unique_references": len(consolidated_search),
        "consolidated_full_text_source_rows": len(full_text_source_rows),
        "consolidated_full_text_unique_references": len(consolidated_texts),
        "targeted_supplement_full_text_rows": len(supplement_text_rows),
        "curated_references_without_stored_text": missing_curated_references,
        "period_groups": evidence["period"].nunique(),
    }
)
audit_path.write_text(
    json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8"
)

consolidation_summary = pd.DataFrame(
    [
        {
            "table": "search query hits",
            "initial_source_rows": len(initial_search_rows),
            "reception_source_rows": len(current_search_rows),
            "supplement_source_rows": 0,
            "combined_source_rows": len(search_source_rows),
            "unique_references": len(consolidated_search),
        },
        {
            "table": "full texts",
            "initial_source_rows": len(initial_text_rows),
            "reception_source_rows": len(current_text_rows),
            "supplement_source_rows": len(supplement_text_rows),
            "combined_source_rows": len(full_text_source_rows),
            "unique_references": len(consolidated_texts),
        },
    ]
)
display(consolidation_summary)
print("Every curated reference has stored Sefaria text.")


,table,initial_source_rows,reception_source_rows,supplement_source_rows,combined_source_rows,unique_references
0,search query hits,119,175,0,294,133
1,full texts,78,325,1,404,332


Every curated reference has stored Sefaria text.


# 10 - Required libraries<a class="anchor" id="bullet10"></a>
##### [Back to ToC](#TOC)

The scripts in this notebook require the following Python libraries to be installed in the environment:

    mcp
    pandas
    tqdm
    nbformat

You can install a missing library from within Jupyter Notebook using either `pip` or `pip3`.

# 11 - Notebook details<a class="anchor" id="bullet11"></a>
##### [Back to ToC](#TOC)

<div style="float: left;">
  <table>
    <tr><td><strong>Author</strong></td><td>Tony Jurg</td></tr>
    <tr><td><strong>Version</strong></td><td>1.2</td></tr>
    <tr><td><strong>Date</strong></td><td>6 August 2026</td></tr>
  </table>
</div>
